# AeroTrack — Monthly Airline Demand Intelligence

## A useful question, with honest limits
Can recent airline-reported passenger counts help an analyst spot route markets that deserve a closer planning review? AeroTrack turns official BTS monthly records into a transparent **route-market watchlist**, showing what was measured, how it was prepared, what models learned, and how well they did on later months.

This is a decision-support prototype for market screening. It does not track planes in real time, identify individual travelers, measure unmet demand, or automatically recommend new service. Before acting, an airline would need schedules, seat supply, fares, bookings, cost, and airport constraints.

## Official source and study window
This project uses U.S. Bureau of Transportation Statistics (BTS), **T-100 Market (All Carriers)**. The table reports passengers by airline, airport pair, and month. The supplied extracts cover January–December 2025 and January–July 2026 (year to date). BTS data is authoritative for the reported measure, but our short window limits generalization.

## Questions AeroTrack answers
1. What does every raw field contain, and where is it missing or unusual?
2. How should route categories and numeric values be prepared for ML?
3. Which signals help estimate passenger counts or identify high-demand records?
4. Which route-month markets should an analyst check first, and how uncertain are the scores?
5. How do simple baselines compare with more complex models on future months held out from training?

## What counts as success
A useful classroom result is reproducible, beats or explains a simple baseline, reports error on later dates, names its limits, and creates a shortlist that can be checked with richer business data. High accuracy alone is not a business outcome.
## Tools used

| Tool | Plain-language job in this project |
|---|---|
| pandas | Open, clean, combine, and summarize tables |
| NumPy | Numeric calculations and calendar features |
| Matplotlib / Seaborn | Draw trends, distributions, and diagnostic plots |
| scikit-learn | Encode, scale, select features, train models, and evaluate them |
| Jupyter | Keep code, explanations, tables, and charts together for review |



In [1]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

PROJECT = Path.cwd()
RAW = PROJECT / "data" / "raw"
FILES = {
    2025: RAW / "bts_t100_market_2025.zip",
    2026: RAW / "bts_t100_market_2026.zip",
}

def read_bts_archive(path):
    with ZipFile(path) as archive:
        csv_files = [name for name in archive.namelist() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError(f"No CSV found inside {path.name}")
        with archive.open(csv_files[0]) as stream:
            return pd.read_csv(stream, low_memory=False)

raw_parts = {year: read_bts_archive(path) for year, path in FILES.items()}
for year, frame in raw_parts.items():
    print(f"{year}: {len(frame):,} rows × {len(frame.columns)} columns")
    print("Months:", sorted(frame["MONTH"].dropna().unique().tolist()))
    print("Passenger total:", f'{frame["PASSENGERS"].sum():,.0f}')
    display(frame.head(3))

2025: 327,350 rows × 10 columns
Months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
Passenger total: 1,102,951,845
   PASSENGERS  DISTANCE UNIQUE_CARRIER  ... DEST  YEAR MONTH
0         0.0       0.0            1BQ  ...  GPT  2025     2
1         0.0       0.0            27Q  ...  OPF  2025    12
2         0.0       0.0            27Q  ...  TEB  2025    11

[3 rows x 10 columns]
2026: 192,052 rows × 10 columns
Months: [1, 2, 3, 4, 5, 6, 7]
Passenger total: 635,655,600
   PASSENGERS  DISTANCE UNIQUE_CARRIER  ... DEST  YEAR MONTH
0         0.0       0.0            27Q  ...  UGN  2026     4
1         0.0       0.0            2NQ  ...  HVN  2026     7
2         0.0       0.0             2O  ...  ADQ  2026     6

[3 rows x 10 columns]


## Field guide

| BTS field | Plain meaning | Project use |
|---|---|---|
| `PASSENGERS` | Reported passengers in the market record | Main prediction target |
| `DISTANCE` | Miles between airports | Numeric predictor |
| `UNIQUE_CARRIER` | Airline code | Categorical predictor |
| `UNIQUE_CARRIER_NAME` | Airline name | Dashboard label |
| `ORIGIN`, `DEST` | Airport codes | Route and categorical predictors |
| `ORIGIN_AIRPORT_ID`, `DEST_AIRPORT_ID` | BTS airport IDs | Stable airport identifiers |
| `YEAR`, `MONTH` | Observation period | Time split and calendar features |

These records are not necessarily a count of distinct people: one traveller can appear in more than one market segment. State that limitation in the final report.


In [2]:
# Check the archive schemas before building any features.
for year, frame in raw_parts.items():
    print(year, frame.columns.tolist())
    assert {"PASSENGERS", "DISTANCE", "UNIQUE_CARRIER", "ORIGIN", "DEST", "YEAR", "MONTH"} <= set(frame.columns)


2025 ['PASSENGERS', 'DISTANCE', 'UNIQUE_CARRIER', 'UNIQUE_CARRIER_NAME', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST', 'YEAR', 'MONTH']
2026 ['PASSENGERS', 'DISTANCE', 'UNIQUE_CARRIER', 'UNIQUE_CARRIER_NAME', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST', 'YEAR', 'MONTH']


## Notebook order
1. Data loading, source context, scope, and limitations.
2. Full-column data profiling, cleaning, missing-value policy, EDA, binning, transformations, and outlier review.
3. Label/one-hot encoding and feature-selection comparison.
4. Simple linear regression.
5. Multiple and polynomial regression.
6. Logistic regression.
7. PCA on Iris (required independent lab exercise).
8. Forecast baselines and dashboard-ready exports.
9. Classification model comparison: Logistic Regression, KNN, Naive Bayes, SVM, and Decision Tree.
10. End-to-end route-market decision-support capstone and watchlist export.

All analysis and model code is in notebooks; there are no project `.py` scripts. Run notebook 01 before later BTS notebooks.
